# Word Embedding — Word2Vec & FastText

Dilakukan training 4 model word embedding dari **seluruh dataset** (7821 ulasan):

| Model | Algoritma | Teks |
|---|---|---|
| word2vec_non_stemmed | Skip-gram | review_non_stemmed |
| word2vec_stemmed | Skip-gram | review_stemmed |
| fasttext_non_stemmed | Skip-gram | review_non_stemmed |
| fasttext_stemmed | Skip-gram | review_stemmed |

Dilatih dari semua data karena jumlah data relatif kecil (~7800 ulasan), melatih embedding dari seluruh data memastikan vocabulary lebih lengkap dan representasi vektor lebih kaya.

**Parameter:**
| Parameter | Nilai | Keterangan |
|---|---|---|
| vector_size | 100 | Dimensi vektor tiap kata |
| window | 5 | Konteks 5 kata kiri dan kanan |
| min_count | 2 | Abaikan kata yang muncul < 2 kali |
| sg | 1 | Skip-gram (lebih baik untuk data kecil) |
| epochs | 10 | Jumlah iterasi training |

---
**Input :** `dataset_for_modeling.csv`  
**Output:** 4 file `.model` tersimpan di Google Drive

## 1. Instalasi & Import Library

In [ ]:
!pip install gensim --quiet

import pandas as pd
import numpy as np
import os
import time
import matplotlib.pyplot as plt
from google.colab import drive
from gensim.models import Word2Vec, FastText

print("Library berhasil diimport.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 30.6 MB/s eta 0:00:00
Library berhasil diimport.


## 2. Mount Google Drive

In [ ]:
drive.mount('/content/drive')

# Path File
INPUT_FILE = '/content/drive/MyDrive/SKRIPSI_NAVY/Dataset/Dataset_Persiapan/Dataset_Text_Preprocessing/dataset_for_modeling.csv'
OUTPUT_DIR = '/content/drive/MyDrive/SKRIPSI_NAVY/Dataset/Dataset_Persiapan/Dataset_WordEmbedding/'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output dir: {OUTPUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output dir: /content/drive/MyDrive/SKRIPSI_NAVY/Dataset/Dataset_Persiapan/Dataset_WordEmbedding/


## 3. Load Dataset & Persiapan Corpus

Persiapan corpus untuk training Word2Vec dan FastText yang berupa **list of list of tokens**, bukan string.

Contoh format:
```
[
  ['kamar', 'bersih', 'nyaman', 'sangat', 'bagus'],
  ['lokasi', 'strategis', 'dekat', 'pusat', 'kota'],
  ...
]
```

In [ ]:
df = pd.read_csv(INPUT_FILE)
df['review_non_stemmed'] = df['review_non_stemmed'].fillna('').astype(str)
df['review_stemmed']     = df['review_stemmed'].fillna('').astype(str)

# Filter baris kosong
df = df[
    (df['review_non_stemmed'].str.strip() != '') &
    (df['review_stemmed'].str.strip() != '')
].reset_index(drop=True)

print(f"Total data: {len(df):,} ulasan")

# Konversi ke list of list of tokens
corpus_non_stemmed = [teks.split() for teks in df['review_non_stemmed']]
corpus_stemmed     = [teks.split() for teks in df['review_stemmed']]

# Statistik corpus
vocab_nonstem = set(token for doc in corpus_non_stemmed for token in doc)
vocab_s  = set(token for doc in corpus_stemmed     for token in doc)

print(f"\nCorpus Non-Stemmed: {len(corpus_non_stemmed):,} dokumen | {len(vocab_nonstem):,} kata unik")
print(f"Corpus Stemmed    : {len(corpus_stemmed):,} dokumen | {len(vocab_s):,} kata unik")
print(f"\nContoh corpus[0] non-stemmed: {corpus_non_stemmed[0]}")
print(f"Contoh corpus[0] stemmed    : {corpus_stemmed[0]}")

Total data: 7,821 ulasan

Corpus Non-Stemmed: 7,821 dokumen | 5,820 kata unik
Corpus Stemmed    : 7,821 dokumen | 3,983 kata unik

Contoh corpus[0] non-stemmed: ['super', 'kagum', 'interior', 'hotel', 'authentic', 'banget', 'kamarnya', 'nyaman', 'televisi', 'besar', 'banget']
Contoh corpus[0] stemmed    : ['super', 'kagum', 'interior', 'hotel', 'authentic', 'banget', 'kamar', 'nyaman', 'televisi', 'besar', 'banget']


## 4. Training Word2Vec

Menggunakan **Algoritma Skip-gram** (`sg=1`): memprediksi kata-kata konteks dari kata target.

### 4a. Word2Vec — Non-Stemmed

In [ ]:
print("Training Word2Vec (Non-Stemmed)...")
start = time.time()

model_w2v_nonstem = Word2Vec(
    sentences   = corpus_non_stemmed,
    vector_size = 100,
    window      = 5,
    min_count   = 2,
    sg          = 1,
    workers     = 4,
    epochs      = 10,
    seed        = 42
)

model_w2v_nonstem.save(OUTPUT_DIR + 'word2vec_non_stemmed.model')

print(f"Selesai dalam   : {time.time() - start:.1f} detik")
print(f"Vocabulary size : {len(model_w2v_nonstem.wv):,} kata")
print(f"Disimpan ke     : {OUTPUT_DIR}word2vec_non_stemmed.model")

Training Word2Vec (Non-Stemmed)...
Selesai dalam   : 4.6 detik
Vocabulary size : 3,411 kata
Disimpan ke     : /content/drive/MyDrive/SKRIPSI_NAVY/Dataset/Dataset_Persiapan/Dataset_WordEmbedding/word2vec_non_stemmed.model


### 4b. Word2Vec — Stemmed

In [ ]:
print("Training Word2Vec (Stemmed)...")
start = time.time()

model_w2v_stem = Word2Vec(
    sentences   = corpus_stemmed,
    vector_size = 100,
    window      = 5,
    min_count   = 2,
    sg          = 1,
    workers     = 4,
    epochs      = 10,
    seed        = 42
)

model_w2v_stem.save(OUTPUT_DIR + 'word2vec_stemmed.model')

print(f"Selesai dalam   : {time.time() - start:.1f} detik")
print(f"Vocabulary size : {len(model_w2v_stem.wv):,} kata")
print(f"Disimpan ke     : {OUTPUT_DIR}word2vec_stemmed.model")

Training Word2Vec (Stemmed)...
Selesai dalam   : 3.5 detik
Vocabulary size : 2,470 kata
Disimpan ke     : /content/drive/MyDrive/SKRIPSI_NAVY/Dataset/Dataset_Persiapan/Dataset_WordEmbedding/word2vec_stemmed.model


## 5. Training FastText

**Perbedaan FastText vs Word2Vec:**

| | Word2Vec | FastText |
|---|---|---|
| Unit belajar | Per kata utuh | Per karakter n-gram (subword) |
| Kata OOV | Tidak bisa | Bisa — vektor diperkirakan dari subword |
| Cocok untuk | Kata baku | Kata informal, typo, variasi penulisan |


### 5a. FastText — Non-Stemmed

In [ ]:
print("Training FastText (Non-Stemmed)...")
start = time.time()

model_ft_nonstem = FastText(
    sentences   = corpus_non_stemmed,
    vector_size = 100,
    window      = 5,
    min_count   = 2,
    sg          = 1,
    workers     = 4,
    epochs      = 10,
    seed        = 42,
    min_n       = 3,
    max_n       = 6
)

model_ft_nonstem.save(OUTPUT_DIR + 'fasttext_non_stemmed.model')

print(f"Selesai dalam   : {time.time() - start:.1f} detik")
print(f"Vocabulary size : {len(model_ft_nonstem.wv):,} kata")
print(f"Disimpan ke     : {OUTPUT_DIR}fasttext_non_stemmed.model")

Training FastText (Non-Stemmed)...
Selesai dalam   : 18.2 detik
Vocabulary size : 3,411 kata
Disimpan ke     : /content/drive/MyDrive/SKRIPSI_NAVY/Dataset/Dataset_Persiapan/Dataset_WordEmbedding/fasttext_non_stemmed.model


### 5b. FastText — Stemmed

In [ ]:
print("Training FastText (Stemmed)...")
start = time.time()

model_ft_stem = FastText(
    sentences   = corpus_stemmed,
    vector_size = 100,
    window      = 5,
    min_count   = 2,
    sg          = 1,
    workers     = 4,
    epochs      = 10,
    seed        = 42,
    min_n       = 3,
    max_n       = 6
)

model_ft_stem.save(OUTPUT_DIR + 'fasttext_stemmed.model')

print(f"Selesai dalam   : {time.time() - start:.1f} detik")
print(f"Vocabulary size : {len(model_ft_stem.wv):,} kata")
print(f"Disimpan ke     : {OUTPUT_DIR}fasttext_stemmed.model")

Training FastText (Stemmed)...
Selesai dalam   : 26.4 detik
Vocabulary size : 2,470 kata
Disimpan ke     : /content/drive/MyDrive/SKRIPSI_NAVY/Dataset/Dataset_Persiapan/Dataset_WordEmbedding/fasttext_stemmed.model


## 6. Cek Output Embedding

Evaluasi kualitas embedding secara kualitatif dengan melihat kata-kata paling mirip secara semantik (`most_similar`) untuk kata kunci domain hotel.

Jika hasilnya masuk akal (contoh: `bersih` mirip dengan `rapi`, `terawat`), maka embedding sudah belajar dengan baik.

In [ ]:
# Fungsi untuk menampilkan kata paling mirip
def cek_similar(model, kata_list, nama_model, topn=5):
    print(f"\n{'='*55}")
    print(f"  {nama_model}")
    print(f"{'='*55}")
    hasil = []
    for kata in kata_list:
        if kata in model.wv:
            similar = model.wv.most_similar(kata, topn=topn)
            similar_str = ', '.join([f"{w} ({s:.3f})" for w, s in similar])
            print(f"  '{kata}' → {similar_str}")
            hasil.append({'Kata': kata, 'Top-5 Kata Mirip': similar_str})
        else:
            print(f"  '{kata}' → tidak ada di vocabulary")
    return pd.DataFrame(hasil)

print("Fungsi siap.")

Fungsi siap.


In [ ]:
# Kata kunci domain hotel untuk evaluasi
kata_uji = ['bersih', 'kotor', 'nyaman', 'ramah', 'lambat',
            'checkin', 'lokasi', 'strategis', 'pelayanan', 'fasilitas']

df_w2v_nonstem = cek_similar(model_w2v_nonstem, kata_uji, 'Word2Vec — Non-Stemmed')
display(df_w2v_nonstem)


  Word2Vec — Non-Stemmed
  'bersih' → harum (0.801), wangi (0.801), rapi (0.794), lobbynya (0.790), tertata (0.766)
  'kotor' → menjijikkan (0.845), kering (0.823), berdebu (0.806), jorok (0.805), lalat (0.796)
  'nyaman' → harum (0.764), empuk (0.745), lobbynya (0.738), lega (0.734), menjamin (0.732)
  'ramah' → friendly (0.815), helpful (0.812), bersahabat (0.804), senyum (0.802), cekatan (0.802)
  'lambat' → internetnya (0.854), terputus (0.830), antre (0.817), kacau (0.809), diabaikan (0.803)
  'checkin' → checkout (0.833), check (0.810), efisien (0.789), pemesanan (0.748), proses (0.742)
  'lokasi' → letaknya (0.837), lokasinya (0.825), posisinya (0.819), letak (0.810), posisi (0.785)
  'strategis' → startegis (0.821), objek (0.805), ideal (0.802), menguntungkan (0.794), monumen (0.786)
  'pelayanan' → pelayanannya (0.821), layanannya (0.789), kerjanya (0.779), housekeeping (0.773), responsif (0.772)
  'fasilitas' → fasilitasnya (0.799), peralatannya (0.726), swimming (0.720), ko

,Kata,Top-5 Kata Mirip
0,bersih,"harum (0.801), wangi (0.801), rapi (0.794), lo..."
1,kotor,"menjijikkan (0.845), kering (0.823), berdebu (..."
2,nyaman,"harum (0.764), empuk (0.745), lobbynya (0.738)..."
3,ramah,"friendly (0.815), helpful (0.812), bersahabat ..."
4,lambat,"internetnya (0.854), terputus (0.830), antre (..."
5,checkin,"checkout (0.833), check (0.810), efisien (0.78..."
6,lokasi,"letaknya (0.837), lokasinya (0.825), posisinya..."
7,strategis,"startegis (0.821), objek (0.805), ideal (0.802..."
8,pelayanan,"pelayanannya (0.821), layanannya (0.789), kerj..."
9,fasilitas,"fasilitasnya (0.799), peralatannya (0.726), sw..."


In [ ]:
# Kata uji untuk versi stemmed (beberapa kata berubah bentuk setelah stemming)
kata_uji_stemmed = ['bersih', 'kotor', 'nyaman', 'ramah', 'lambat',
                    'checkin', 'lokasi', 'strategi', 'layan', 'fasilitas']

df_w2v_stem = cek_similar(model_w2v_stem, kata_uji_stemmed, 'Word2Vec — Stemmed')
display(df_w2v_stem)


  Word2Vec — Stemmed
  'bersih' → wangi (0.752), lobbynya (0.749), kinclong (0.746), lega (0.727), rapi (0.711)
  'kotor' → noda (0.857), jijik (0.837), kecoak (0.828), debu (0.823), kering (0.821)
  'nyaman' → jamin (0.812), lega (0.773), privat (0.769), empuk (0.758), lobbynya (0.754)
  'ramah' → friendly (0.824), sahabat (0.812), cekat (0.799), ketemu (0.797), helpful (0.792)
  'lambat' → antre (0.815), internetnya (0.804), putus (0.794), lamban (0.789), kacau (0.784)
  'checkin' → checkout (0.812), check (0.801), efisien (0.794), chek (0.757), proses (0.747)
  'lokasi' → posisi (0.806), objek (0.799), kawasan (0.790), mobilisasi (0.784), km (0.774)
  'strategi' → slot (0.961), wkwk (0.960), ongkos (0.959), kenongo (0.959), keman (0.958)
  'layan' → responsif (0.820), cekat (0.785), terap (0.780), minta (0.774), sahabat (0.773)
  'fasilitas' → gym (0.710), musala (0.709), center (0.691), spa (0.686), swimming (0.685)


,Kata,Top-5 Kata Mirip
0,bersih,"wangi (0.752), lobbynya (0.749), kinclong (0.7..."
1,kotor,"noda (0.857), jijik (0.837), kecoak (0.828), d..."
2,nyaman,"jamin (0.812), lega (0.773), privat (0.769), e..."
3,ramah,"friendly (0.824), sahabat (0.812), cekat (0.79..."
4,lambat,"antre (0.815), internetnya (0.804), putus (0.7..."
5,checkin,"checkout (0.812), check (0.801), efisien (0.79..."
6,lokasi,"posisi (0.806), objek (0.799), kawasan (0.790)..."
7,strategi,"slot (0.961), wkwk (0.960), ongkos (0.959), ke..."
8,layan,"responsif (0.820), cekat (0.785), terap (0.780..."
9,fasilitas,"gym (0.710), musala (0.709), center (0.691), s..."


In [ ]:
df_ft_nonstem = cek_similar(model_ft_nonstem, kata_uji, 'FastText — Non-Stemmed')
display(df_ft_nonstem)


  FastText — Non-Stemmed
  'bersih' → rapi (0.823), kebersihannya (0.800), pembersih (0.786), empuk (0.774), kinclong (0.771)
  'kotor' → kotoran (0.919), jijik (0.876), kecoa (0.875), kecoak (0.872), menjijikkan (0.871)
  'nyaman' → ternyaman (0.891), nyamannya (0.866), empuk (0.815), nyenyak (0.788), sejuk (0.781)
  'ramah' → ramahnya (0.878), tamah (0.851), bersahabat (0.823), senyum (0.820), helpful (0.819)
  'lambat' → lamban (0.921), terlambat (0.886), kacau (0.861), terhambat (0.860), obat (0.852)
  'checkin' → check (0.985), checkout (0.937), earlycheckin (0.931), chek (0.925), efisien (0.863)
  'lokasi' → lokasinya (0.933), berlokasi (0.921), letaknya (0.867), posisi (0.852), posisinya (0.851)
  'strategis' → strategi (0.988), stategis (0.969), startegis (0.909), objek (0.861), homey (0.851)
  'pelayanan' → pelayan (0.975), pelayanannya (0.934), pelayananya (0.923), layanan (0.916), layanannya (0.877)
  'fasilitas' → fasilitasnya (0.936), disabilitas (0.829), kualitasnya (0.7

,Kata,Top-5 Kata Mirip
0,bersih,"rapi (0.823), kebersihannya (0.800), pembersih..."
1,kotor,"kotoran (0.919), jijik (0.876), kecoa (0.875),..."
2,nyaman,"ternyaman (0.891), nyamannya (0.866), empuk (0..."
3,ramah,"ramahnya (0.878), tamah (0.851), bersahabat (0..."
4,lambat,"lamban (0.921), terlambat (0.886), kacau (0.86..."
5,checkin,"check (0.985), checkout (0.937), earlycheckin ..."
6,lokasi,"lokasinya (0.933), berlokasi (0.921), letaknya..."
7,strategis,"strategi (0.988), stategis (0.969), startegis ..."
8,pelayanan,"pelayan (0.975), pelayanannya (0.934), pelayan..."
9,fasilitas,"fasilitasnya (0.936), disabilitas (0.829), kua..."


In [ ]:
df_ft_stem = cek_similar(model_ft_stem, kata_uji_stemmed, 'FastText — Stemmed')
display(df_ft_stem)


  FastText — Stemmed
  'bersih' → ber (0.782), rapi (0.769), kinclong (0.749), amenitiesnya (0.741), kondisioner (0.736)
  'kotor' → jijik (0.896), kecoak (0.881), kecoa (0.875), lumut (0.861), debu (0.859)
  'nyaman' → jamin (0.827), jaman (0.816), empuk (0.798), amin (0.785), leluasa (0.777)
  'ramah' → tamah (0.829), suportif (0.826), manis (0.825), keramahtamahannya (0.824), cekat (0.816)
  'lambat' → lamban (0.952), hambat (0.946), obat (0.828), antre (0.827), kacau (0.823)
  'checkin' → check (0.984), checkout (0.948), chek (0.924), earlycheckin (0.918), latecheckout (0.860)
  'lokasi' → posisi (0.839), objek (0.831), homey (0.822), daerah (0.806), kawasan (0.804)
  'strategi' → strategis (0.989), stategis (0.973), startegis (0.912), street (0.861), objek (0.860)
  'layan' → pelayananya (0.859), responsif (0.853), swalayan (0.840), respons (0.829), apresiasi (0.825)
  'fasilitas' → disabilitas (0.836), fungsionalitas (0.796), totalitas (0.796), musala (0.784), balita (0.784)


,Kata,Top-5 Kata Mirip
0,bersih,"ber (0.782), rapi (0.769), kinclong (0.749), a..."
1,kotor,"jijik (0.896), kecoak (0.881), kecoa (0.875), ..."
2,nyaman,"jamin (0.827), jaman (0.816), empuk (0.798), a..."
3,ramah,"tamah (0.829), suportif (0.826), manis (0.825)..."
4,lambat,"lamban (0.952), hambat (0.946), obat (0.828), ..."
5,checkin,"check (0.984), checkout (0.948), chek (0.924),..."
6,lokasi,"posisi (0.839), objek (0.831), homey (0.822), ..."
7,strategi,"strategis (0.989), stategis (0.973), startegis..."
8,layan,"pelayananya (0.859), responsif (0.853), swalay..."
9,fasilitas,"disabilitas (0.836), fungsionalitas (0.796), t..."


### Uji OOV FastText

Salah satu keunggulan FastText adalah kemampuannya menghasilkan vektor untuk kata yang **tidak ada di vocabulary training** (Out-of-Vocabulary / OOV), menggunakan informasi subword (karakter n-gram).

In [ ]:
print("Uji kemampuan FastText menangani kata OOV:")
print("-" * 55)

kata_oov = ['kebersihann', 'nyamann', 'kotr', 'stategis', 'recomended']

for kata in kata_oov:
    ada_di_vocab = kata in model_ft_nonstem.wv.key_to_index
    similar      = model_ft_nonstem.wv.most_similar(kata, topn=3)
    similar_str  = ', '.join([f"{w} ({s:.3f})" for w, s in similar])
    print(f"  '{kata}'")
    print(f"    Ada di vocab : {ada_di_vocab}")
    print(f"    Mirip dengan : {similar_str}")
    print()

Uji kemampuan FastText menangani kata OOV:
-------------------------------------------------------
  'kebersihann'
    Ada di vocab : False
    Mirip dengan : kebersihan (0.985), kebersihannya (0.954), pembersihan (0.884)

  'nyamann'
    Ada di vocab : False
    Mirip dengan : nyaman (0.966), nyamannya (0.883), ternyaman (0.843)

  'kotr'
    Ada di vocab : False
    Mirip dengan : kotor (0.938), kotak (0.926), kotoran (0.844)

  'stategis'
    Ada di vocab : True
    Mirip dengan : strategi (0.973), strategis (0.969), startegis (0.957)

  'recomended'
    Ada di vocab : True
    Mirip dengan : recomend (0.993), recommended (0.992), recommend (0.981)



## 7. Ringkasan & Verifikasi

In [ ]:
print("=" * 55)
print("        RINGKASAN WORD EMBEDDING")
print("=" * 55)
print(f"Data training : {len(df):,} ulasan (seluruh dataset)")
print(f"Algoritma     : Skip-gram (sg=1)")
print(f"vector_size   : 100")
print(f"window        : 5")
print(f"min_count     : 2")
print(f"epochs        : 10")
print()

models_info = [
    ('Word2Vec Non-Stemmed', model_w2v_nonstem, 'word2vec_non_stemmed.model'),
    ('Word2Vec Stemmed',     model_w2v_stem,  'word2vec_stemmed.model'),
    ('FastText Non-Stemmed', model_ft_nonstem,  'fasttext_non_stemmed.model'),
    ('FastText Stemmed',     model_ft_stem,   'fasttext_stemmed.model'),
]

print(f"{'Model':<25} {'Vocab':>8}   File")
print("-" * 55)
for nama, model, fname in models_info:
    print(f"  {nama:<23} {len(model.wv):>8,}   {fname}")

print()
print("Verifikasi file tersimpan:")
for _, _, fname in models_info:
    path   = OUTPUT_DIR + fname
    exists = os.path.exists(path)
    size   = os.path.getsize(path)/1024/1024 if exists else 0
    status = f"OK ({size:.1f} MB)" if exists else "TIDAK ADA!"
    print(f"  [{status}] {fname}")

        RINGKASAN WORD EMBEDDING
Data training : 7,821 ulasan (seluruh dataset)
Algoritma     : Skip-gram (sg=1)
vector_size   : 100
window        : 5
min_count     : 2
epochs        : 10

Model                        Vocab   File
-------------------------------------------------------
  Word2Vec Non-Stemmed       3,411   word2vec_non_stemmed.model
  Word2Vec Stemmed           2,470   word2vec_stemmed.model
  FastText Non-Stemmed       3,411   fasttext_non_stemmed.model
  FastText Stemmed           2,470   fasttext_stemmed.model

Verifikasi file tersimpan:
  [OK (2.7 MB)] word2vec_non_stemmed.model
  [OK (2.0 MB)] word2vec_stemmed.model
  [OK (2.7 MB)] fasttext_non_stemmed.model
  [OK (2.0 MB)] fasttext_stemmed.model
